# 01 First EDA (preliminary)

**Owner:** ER · **Date:** 2026-11-03

**Purpose:** first descriptive look at the prototype week for the concept presentation (issue #12). No tests, no models.

**Input:** table `analysis_base` in the project database (built with `python -m src.build_db --days 2026-01-05 2026-01-11`), week **05.–11.01.2026**, canton of Zurich.

**Output:** tables in this notebook and three figures in `presentation/figures/` (all marked *preliminary*).

> One week is far too little for conclusions: all arrivals in the same hour share the same weather, and single days dominate. Everything here is **preliminary**.

Setup: project paths and database helpers from `src/` (same code for SQLite and PostgreSQL).

In [ ]:
import sys; sys.path.append("..")
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from src.paths import RAW_DIR, PROCESSED_DIR, ROOT
from src.db import query, write_table, list_tables, backend

FIG_DIR = ROOT / "presentation" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Fixed colours for the whole project: regional = blue, long-distance = orange
# (checked for colour-vision deficiency with a palette validator)
COLORS = {"regional": "#2a78d6", "long_distance": "#eb6834"}
LABELS = {"regional": "Regional (S-Bahn)", "long_distance": "Long-distance (IC, IR, RE, EC ...)"}
SOURCE = "Source: opentransportdata.swiss (Ist-Daten), MeteoSwiss; own database. Week 05.-11.01.2026, canton ZH."

# Recessive axes and grid for slide figures
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e6e6", "axes.axisbelow": True, "font.size": 11})
print("database backend:", backend())

## 1. Load data

What is in the database (`list_tables()` is also the basis for the slide *data structure*).

In [ ]:
list_tables()

The analysis table is built in SQL (joins in `src/analysis_base.sql`). Here we only read it.

In [ ]:
df = query("SELECT * FROM analysis_base")
print(f"{len(df):,} arrivals, {df.shape[1]} columns, {df.local_date.nunique()} local dates")
df.head()

## 2. Data structure

Columns, data types and missing values. Missing `entry_delay_min` is expected: it is only defined *after* the first stop of a trip in the study area.

In [ ]:
structure = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_pct": (100 * df.isna().mean()).round(1),
    "example": df.iloc[0],
})
structure

Size of the groups we compare (traffic type × weather), counted **in SQL** with `GROUP BY`. `hours` = number of distinct weather hours behind each group: that is the real amount of independent weather information.

In [ ]:
groups = query('''
    SELECT traffic_type, precip_cat,
           COUNT(*)                         AS arrivals,
           SUM(delayed)                     AS delayed,
           ROUND(100.0 * AVG(delayed), 1)   AS delay_rate_pct,
           COUNT(DISTINCT weather_hour_utc) AS hours
    FROM analysis_base
    GROUP BY traffic_type, precip_cat
    ORDER BY traffic_type, precip_cat
''')
groups

## 3. Delay rate per traffic type × dry / rain / snow

Delayed = arrival ≥ 3 min late (SBB definition). Bars = share of delayed arrivals, label = rate and number of arrivals.

In [ ]:
cats = ["dry", "rain", "snow"]
fig, ax = plt.subplots(figsize=(9, 5))
width = 0.38
for i, tt in enumerate(["regional", "long_distance"]):
    g = groups[groups.traffic_type == tt].set_index("precip_cat").reindex(cats)
    # bar positions: two bars per weather class with a small gap between them
    x = [c + (i - 0.5) * (width + 0.02) for c in range(len(cats))]
    ax.bar(x, g.delay_rate_pct, width=width, color=COLORS[tt], label=LABELS[tt])
    for xi, rate, n in zip(x, g.delay_rate_pct, g.arrivals):
        ax.text(xi, rate + 0.3, f"{rate:.1f} %\nn={n:,}", ha="center", va="bottom", fontsize=9, color="#333333")
ax.set_xticks(range(len(cats)), ["Dry", "Rain", "Snow"])
ax.set_ylabel("Arrivals >= 3 min late (%)")
ax.set_ylim(0, groups.delay_rate_pct.max() * 1.4)
ax.legend(frameon=False, loc="upper left")
ax.set_title("PRELIMINARY (1 week): no clear weather pattern yet in the delay rate", loc="left", fontsize=13)
fig.text(0.01, -0.02, SOURCE + " Weather = last full hour before the scheduled arrival.", fontsize=8, color="#555555")
fig.tight_layout()
fig.savefig(FIG_DIR / "prelim_delay_rate_by_weather.png", dpi=200, bbox_inches="tight")
plt.show()

Main comparison decided for the project: precipitation (rain + snow) vs. dry, per traffic type. Absolute difference in percentage points (descriptive only, no test).

In [ ]:
main = query('''
    SELECT traffic_type, precip_any, COUNT(*) AS arrivals, ROUND(100.0 * AVG(delayed), 2) AS delay_rate_pct
    FROM analysis_base GROUP BY traffic_type, precip_any
''').pivot(index="traffic_type", columns="precip_any", values="delay_rate_pct")
main.columns = ["dry_pct", "precipitation_pct"]
main["difference_pp"] = (main.precipitation_pct - main.dry_pct).round(2)
main

## 4. Distribution of the arrival delays

Most trains arrive about one minute late; the 3-minute threshold cuts off the right tail. Values outside −2 … 15 min are grouped at the edges.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
bins = [x / 2 for x in range(-4, 31)]  # half-minute bins from -2 to 15 min
for ax, tt in zip(axes, ["regional", "long_distance"]):
    d = df.loc[df.traffic_type == tt, "arr_delay_min"].clip(-2, 15)
    # weights turn counts into percent of arrivals, so both panels are comparable
    ax.hist(d, bins=bins, color=COLORS[tt], weights=[100 / len(d)] * len(d), edgecolor="white", linewidth=0.5)
    ax.axvline(3, color="#333333", lw=1.2, ls="--")
    ax.set_title(f"{LABELS[tt]}\nn={len(d):,}, median {d.median():.1f} min", fontsize=11, loc="left")
    ax.set_xlabel("Arrival delay (min), clipped to -2 ... 15")
for ax in axes:
    ax.text(3.2, ax.get_ylim()[1] * 0.92, "3 min threshold", fontsize=9)
axes[0].set_ylabel("Share of arrivals (%)")
fig.suptitle("PRELIMINARY (1 week): most arrivals are about 1 min late; only the tail passes 3 min",
             x=0.01, ha="left", fontsize=13)
fig.text(0.01, -0.03, SOURCE, fontsize=8, color="#555555")
fig.tight_layout()
fig.savefig(FIG_DIR / "prelim_delay_distribution.png", dpi=200, bbox_inches="tight")
plt.show()

## 5. Delay rate per day

Why one week is not enough: single days move the rate more than the weather does. Aggregated in SQL per local date.

In [ ]:
daily = query('''
    SELECT local_date, traffic_type, COUNT(*) AS arrivals, ROUND(100.0 * AVG(delayed), 1) AS delay_rate_pct,
           ROUND(100.0 * AVG(precip_any), 1) AS share_with_precip_pct
    FROM analysis_base
    WHERE local_date BETWEEN '2026-01-05' AND '2026-01-11'
    GROUP BY local_date, traffic_type ORDER BY local_date
''')
fig, ax = plt.subplots(figsize=(9, 4.5))
for tt in ["regional", "long_distance"]:
    g = daily[daily.traffic_type == tt]
    ax.plot(pd.to_datetime(g.local_date), g.delay_rate_pct, marker="o", ms=8, lw=2, color=COLORS[tt], label=LABELS[tt])
ax.set_ylabel("Arrivals >= 3 min late (%)")
ax.set_ylim(0, daily.delay_rate_pct.max() * 1.15)
ax.legend(frameon=False)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%a %d.%m."))
ax.set_title("PRELIMINARY: day-to-day differences are larger than weather differences", loc="left", fontsize=13)
fig.text(0.01, -0.03, SOURCE, fontsize=8, color="#555555")
fig.tight_layout()
fig.savefig(FIG_DIR / "prelim_delay_rate_by_day.png", dpi=200, bbox_inches="tight")
plt.show()
daily.pivot(index="local_date", columns="traffic_type", values=["delay_rate_pct", "share_with_precip_pct"])

## Findings

*(for the slides, all preliminary, one week 05.–11.01.2026)*

- The data pipeline works end-to-end: every arrival in the week got a precipitation and a temperature value (0 missing).
- Regional traffic (S-Bahn) has far more arrivals than long-distance traffic; long-distance traffic has the higher delay rate in every weather class.
- **No clear weather pattern yet:** in this week the S-Bahn delay rate with precipitation is about the same as without; for long-distance trains it is slightly higher.
- **Day effects dominate** (section 5). This is why we need the whole study period (15 months) and the control variables (hour, weekday, holiday, month, winter).

## Open questions

- Snow comes from few weather hours (column `hours`); arrivals in the same hour are not independent → effect sizes with confidence intervals at hour or day level later (hypothesis tests are a separate issue).
- What happened on Tue 06.01.2026 (dry, but high delay rate)? Check before interpreting anything.
- Night S-Bahn (`SN`) and Railjet (`RJ`) are excluded for now → decision pending.

### Jupyter notebook --footer info--

In [ ]:
import os, platform
from platform import python_version
from datetime import datetime
print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')